# Day 5 · Notebook 1 of 5 – Dashboard views (in the Gold schema)

**What this notebook does**
1. Deletes the old `capstone_5iot.dashboard` schema. It is no longer used.
2. Creates 9 views directly in `capstone_5iot.gold`. Each view feeds one page or widget of the AI/BI dashboard.
3. Checks that every view returns rows.
4. Lists the dashboard layout to build in the Dashboards UI.

**Why views in Gold?** Gold is already the business-ready layer. The views only add presentation logic (totals, ranks, Pareto %, drift flags), so they live next to the Gold tables with a `vw_` prefix. Analysts who have SELECT on Gold (Day 4 grants) can automatically read these views.

**Day 5 run order:** `7_1_dashboard_views` → `7_2_final_regression` → `7_3_uat_signoff` → `7_4_ml_mlflow` → `7_5_genie_runbook_signoff`

**KPI definitions used in the views**
* **Availability** = (1,440 − downtime minutes) / 1,440 per machine-day. For a group or period: (machines × days × 1,440 − downtime) / (machines × days × 1,440).
* **Abnormal reading** = temperature > 85 °C OR vibration > 7.1 mm/s OR pressure > 18 bar. **Abnormal rate** = abnormal / valid readings.
* **MTBF (hours)** = (machines × days × 24 − downtime hours) / failures.
* **MTTR (minutes)** = downtime minutes / failures.

## Step 1 – Delete the old `dashboard` schema
`CASCADE` also deletes the views inside it. Nothing else uses this schema. If you already built a dashboard or Genie space on the old views, point their datasets to `capstone_5iot.gold.vw_...` instead.

In [0]:
%sql
DROP SCHEMA IF EXISTS capstone_5iot.dashboard CASCADE;

## Step 2 – Create the views

### 2a. `vw_kpi_headline` – one row with the headline KPIs
Used for the counter tiles on the first dashboard page.

In [0]:
%sql
CREATE OR REPLACE VIEW capstone_5iot.gold.vw_kpi_headline
COMMENT 'Dashboard: headline KPIs - availability, abnormal rate, downtime, failures, MTBF, MTTR'
AS
WITH totals AS (
  SELECT sum(reading_count)    AS readings,
         sum(abnormal_count)   AS abnormal,
         sum(failure_count)    AS failures,
         sum(downtime_minutes) AS downtime,
         min(activity_date)    AS first_day,
         max(activity_date)    AS last_day
  FROM capstone_5iot.gold.mart_sensor_daily
),
sizes AS (
  SELECT (SELECT count(*) FROM capstone_5iot.gold.dim_machine) AS machines,
         (SELECT count(*) FROM capstone_5iot.gold.dim_date)    AS days
),
repairs AS (
  SELECT avg(repair_hours)          AS repair_hours,
         count(DISTINCT machine_id) AS failing_machines
  FROM capstone_5iot.gold.fact_maintenance_episodes
)
SELECT t.first_day                                   AS period_start,
       t.last_day                                    AS period_end,
       s.days                                        AS days_in_period,
       s.machines                                    AS machines_monitored,
       r.failing_machines                            AS machines_with_failures,
       t.readings                                    AS valid_readings,
       t.abnormal                                    AS abnormal_readings,
       round(t.abnormal / t.readings, 4)             AS abnormal_reading_rate,
       t.failures                                    AS failure_count,
       t.downtime                                    AS total_downtime_minutes,
       round((s.machines * s.days * 1440 - t.downtime) / (s.machines * s.days * 1440), 6) AS overall_availability,
       round((s.machines * s.days * 24 - t.downtime / 60) / nullif(t.failures, 0), 1)     AS mtbf_hours,
       round(t.downtime / nullif(t.failures, 0), 2)  AS mttr_downtime_minutes_per_failure,
       round(r.repair_hours, 1)                      AS mean_hours_failure_to_repair_complete
FROM totals t CROSS JOIN sizes s CROSS JOIN repairs r;

### 2b. `vw_daily_trend` – one row per day for the whole fleet
Used for the line and bar charts over time. Days with no data still appear (taken from `dim_date`).

In [0]:
%sql
CREATE OR REPLACE VIEW capstone_5iot.gold.vw_daily_trend
COMMENT 'Dashboard: per day - fleet availability, abnormal rate, downtime, failures'
AS
WITH fleet AS (SELECT count(*) AS machines FROM capstone_5iot.gold.dim_machine)
SELECT d.date                                     AS activity_date,
       d.day_name,
       count(m.machine_id)                        AS machines_reporting,
       coalesce(sum(m.reading_count), 0)          AS valid_readings,
       coalesce(sum(m.abnormal_count), 0)         AS abnormal_readings,
       round(sum(m.abnormal_count) / nullif(sum(m.reading_count), 0), 4) AS abnormal_reading_rate,
       coalesce(sum(m.failure_count), 0)          AS failures,
       coalesce(sum(m.downtime_minutes), 0)       AS downtime_minutes,
       round((max(f.machines) * 1440 - coalesce(sum(m.downtime_minutes), 0)) / (max(f.machines) * 1440), 6) AS fleet_availability
FROM capstone_5iot.gold.dim_date d
CROSS JOIN fleet f
LEFT JOIN capstone_5iot.gold.mart_sensor_daily m ON m.activity_date = d.date
GROUP BY d.date, d.day_name;

### 2c. `vw_machine_reliability` – one row per machine
Machine attributes, plant, failures, downtime, availability and a downtime rank (1 = most downtime).

In [0]:
%sql
CREATE OR REPLACE VIEW capstone_5iot.gold.vw_machine_reliability
COMMENT 'Dashboard: per machine - plant, type, criticality, failures, downtime, availability, abnormal rate, downtime rank'
AS
WITH period AS (SELECT count(*) AS days FROM capstone_5iot.gold.dim_date),
per_machine AS (
  SELECT machine_id,
         sum(reading_count)    AS readings,
         sum(abnormal_count)   AS abnormal,
         sum(failure_count)    AS failures,
         sum(downtime_minutes) AS downtime
  FROM capstone_5iot.gold.mart_sensor_daily
  GROUP BY machine_id
)
SELECT d.machine_id, d.machine_type, d.manufacturer, d.criticality, d.operating_status, d.line_id,
       d.plant_id, p.plant_name, p.region, p.country,
       coalesce(a.readings, 0)                         AS valid_readings,
       coalesce(a.abnormal, 0)                         AS abnormal_readings,
       round(a.abnormal / nullif(a.readings, 0), 4)    AS abnormal_reading_rate,
       coalesce(a.failures, 0)                         AS failures,
       coalesce(a.downtime, 0)                         AS downtime_minutes,
       round((k.days * 1440 - coalesce(a.downtime, 0)) / (k.days * 1440), 6) AS availability,
       rank() OVER (ORDER BY coalesce(a.downtime, 0) DESC) AS downtime_rank
FROM capstone_5iot.gold.dim_machine d
CROSS JOIN period k
LEFT JOIN capstone_5iot.gold.dim_plant p ON d.plant_sk = p.plant_sk
LEFT JOIN per_machine a ON a.machine_id = d.machine_id;

### 2d. `vw_plant_summary` – one row per plant
Machines, failures, downtime and availability per plant and region. Plants with no machines show availability as NULL.

In [0]:
%sql
CREATE OR REPLACE VIEW capstone_5iot.gold.vw_plant_summary
COMMENT 'Dashboard: per plant and region - machines, failures, downtime, availability'
AS
WITH period AS (SELECT count(*) AS days FROM capstone_5iot.gold.dim_date),
machine_counts AS (
  SELECT plant_id, count(*) AS machines, count_if(criticality = 'HIGH') AS high_crit
  FROM capstone_5iot.gold.dim_machine
  GROUP BY plant_id
),
per_plant AS (
  SELECT plant_id,
         sum(reading_count)    AS readings,
         sum(abnormal_count)   AS abnormal,
         sum(failure_count)    AS failures,
         sum(downtime_minutes) AS downtime
  FROM capstone_5iot.gold.mart_sensor_daily
  GROUP BY plant_id
)
SELECT p.plant_id, p.plant_name, p.region, p.country, p.plant_type,
       coalesce(mc.machines, 0)                      AS machines,
       coalesce(mc.high_crit, 0)                     AS high_criticality_machines,
       coalesce(a.readings, 0)                       AS valid_readings,
       round(a.abnormal / nullif(a.readings, 0), 4)  AS abnormal_reading_rate,
       coalesce(a.failures, 0)                       AS failures,
       coalesce(a.downtime, 0)                       AS downtime_minutes,
       CASE WHEN mc.machines > 0
            THEN round((mc.machines * k.days * 1440 - coalesce(a.downtime, 0)) / (mc.machines * k.days * 1440), 6)
       END                                           AS availability
FROM capstone_5iot.gold.dim_plant p
CROSS JOIN period k
LEFT JOIN machine_counts mc ON mc.plant_id = p.plant_id
LEFT JOIN per_plant a      ON a.plant_id  = p.plant_id;

### 2e. `vw_failure_pareto` – one row per failure code
Codes are sorted by downtime. A code is in the **"Vital few"** group while the downtime *before* it is still under 80%, so the code that crosses 80% is included.

In [0]:
%sql
CREATE OR REPLACE VIEW capstone_5iot.gold.vw_failure_pareto
COMMENT 'Dashboard: failures and downtime by failure code with cumulative percent (Pareto)'
AS
WITH by_code AS (
  SELECT c.failure_code, c.failure_description, c.failure_category, c.root_cause_group, c.severity_default,
         count(*)               AS failures,
         sum(e.downtime_minutes) AS downtime_minutes
  FROM capstone_5iot.gold.fact_maintenance_episodes e
  JOIN capstone_5iot.gold.dim_failure_code c ON e.failure_code_sk = c.failure_code_sk
  GROUP BY ALL
),
running AS (
  SELECT *,
         row_number() OVER (ORDER BY downtime_minutes DESC, failures DESC, failure_code) AS pareto_rank,
         sum(downtime_minutes) OVER (ORDER BY downtime_minutes DESC, failures DESC, failure_code
                                     ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cum_downtime,
         sum(failures)         OVER (ORDER BY downtime_minutes DESC, failures DESC, failure_code
                                     ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cum_failures,
         sum(downtime_minutes) OVER () AS total_downtime,
         sum(failures)         OVER () AS total_failures
  FROM by_code
)
SELECT failure_code, failure_description, failure_category, root_cause_group, severity_default,
       failures, downtime_minutes, pareto_rank,
       round(100 * downtime_minutes / total_downtime, 2) AS downtime_pct,
       round(100 * cum_downtime / total_downtime, 2)     AS cumulative_downtime_pct,
       round(100 * cum_failures / total_failures, 2)     AS cumulative_failure_pct,
       CASE WHEN cum_downtime - downtime_minutes < 0.8 * total_downtime
            THEN 'Vital few (80% of downtime)' ELSE 'Useful many' END AS pareto_class
FROM running;

### 2f. `vw_sensor_drift` – one row per machine per day
Compares each day's average with the **previous 7 days** of the same machine (today is not part of the baseline).
A drift flag is raised when there are at least 3 baseline days and the value is more than **2 standard deviations** away from the 7-day average.

In [0]:
%sql
CREATE OR REPLACE VIEW capstone_5iot.gold.vw_sensor_drift
COMMENT 'Dashboard: per machine per day - sensor averages, previous 7-day rolling mean and 2-sigma drift flags'
AS
WITH daily AS (
  SELECT machine_id, plant_id, to_date(reading_ts) AS activity_date,
         count(*)              AS readings,
         count_if(is_abnormal) AS abnormal_readings,
         avg(temperature_c) AS avg_temperature_c, max(temperature_c) AS max_temperature_c,
         avg(vibration_mm_s) AS avg_vibration_mm_s, max(vibration_mm_s) AS max_vibration_mm_s,
         avg(pressure_bar)   AS avg_pressure_bar,   max(pressure_bar)   AS max_pressure_bar
  FROM capstone_5iot.gold.fact_sensor_activity
  GROUP BY ALL
),
baseline AS (
  SELECT *,
         count(*)                    OVER w7 AS baseline_days,
         avg(avg_temperature_c)      OVER w7 AS base_temp,  stddev_samp(avg_temperature_c)  OVER w7 AS std_temp,
         avg(avg_vibration_mm_s)     OVER w7 AS base_vib,   stddev_samp(avg_vibration_mm_s) OVER w7 AS std_vib,
         avg(avg_pressure_bar)       OVER w7 AS base_pres,  stddev_samp(avg_pressure_bar)   OVER w7 AS std_pres
  FROM daily
  WINDOW w7 AS (PARTITION BY machine_id ORDER BY unix_date(activity_date) RANGE BETWEEN 7 PRECEDING AND 1 PRECEDING)
),
flags AS (
  SELECT *,
         coalesce(baseline_days >= 3 AND std_temp > 0 AND abs(avg_temperature_c - base_temp) > 2 * std_temp, false) AS temperature_drift_flag,
         coalesce(baseline_days >= 3 AND std_vib  > 0 AND abs(avg_vibration_mm_s - base_vib) > 2 * std_vib,  false) AS vibration_drift_flag,
         coalesce(baseline_days >= 3 AND std_pres > 0 AND abs(avg_pressure_bar - base_pres)  > 2 * std_pres, false) AS pressure_drift_flag
  FROM baseline
)
SELECT machine_id, plant_id, activity_date, readings, abnormal_readings,
       round(avg_temperature_c, 2)  AS avg_temperature_c,  round(max_temperature_c, 2)  AS max_temperature_c,
       round(avg_vibration_mm_s, 3) AS avg_vibration_mm_s, round(max_vibration_mm_s, 3) AS max_vibration_mm_s,
       round(avg_pressure_bar, 2)   AS avg_pressure_bar,   round(max_pressure_bar, 2)   AS max_pressure_bar,
       round(base_temp, 2) AS rolling_7d_avg_temperature_c,
       round(base_vib, 3)  AS rolling_7d_avg_vibration_mm_s,
       round(base_pres, 2) AS rolling_7d_avg_pressure_bar,
       baseline_days, temperature_drift_flag, vibration_drift_flag, pressure_drift_flag,
       temperature_drift_flag OR vibration_drift_flag OR pressure_drift_flag AS any_drift_flag
FROM flags;

### 2g. `vw_failure_correlation` – are readings worse before a failure?
Every valid reading goes into one of three groups:
1. within 24 hours before a failure of the same machine,
2. other readings of machines that had a failure,
3. readings of machines that never failed.

`lift_vs_no_failure_machines` = the group's abnormal rate ÷ group 3's abnormal rate. A lift above 1 means more abnormal readings than normal.

In [0]:
%sql
CREATE OR REPLACE VIEW capstone_5iot.gold.vw_failure_correlation
COMMENT 'Dashboard: abnormal readings in the 24h before failures vs machines without failures'
AS
WITH failed_machines AS (
  SELECT DISTINCT machine_id FROM capstone_5iot.gold.fact_maintenance_episodes
),
before_failure AS (
  SELECT DISTINCT r.reading_id
  FROM capstone_5iot.gold.fact_sensor_activity r
  JOIN capstone_5iot.gold.fact_maintenance_episodes f
    ON r.machine_id = f.machine_id
   AND r.reading_ts <= f.failure_ts
   AND r.reading_ts >  f.failure_ts - INTERVAL 24 HOURS
),
grouped AS (
  SELECT r.*,
         CASE WHEN b.reading_id IS NOT NULL THEN '1. Within 24h before a failure'
              WHEN fm.machine_id IS NOT NULL THEN '2. Other readings, machines with failures'
              ELSE '3. Machines with no failure' END AS reading_group
  FROM capstone_5iot.gold.fact_sensor_activity r
  LEFT JOIN before_failure b   ON r.reading_id = b.reading_id
  LEFT JOIN failed_machines fm ON r.machine_id = fm.machine_id
),
summary AS (
  SELECT reading_group,
         count(DISTINCT machine_id)                     AS machines,
         count(*)                                       AS readings,
         count_if(is_abnormal)                          AS abnormal_readings,
         round(count_if(is_abnormal) / count(*), 4)     AS abnormal_reading_rate,
         round(avg(temperature_c), 2)                   AS avg_temperature_c,
         round(avg(vibration_mm_s), 3)                  AS avg_vibration_mm_s,
         round(avg(pressure_bar), 2)                    AS avg_pressure_bar
  FROM grouped
  GROUP BY reading_group
)
SELECT *,
       round(abnormal_reading_rate /
             max(CASE WHEN reading_group LIKE '3.%' THEN abnormal_reading_rate END) OVER (), 2) AS lift_vs_no_failure_machines
FROM summary;

### 2h. `vw_maintenance` – work orders by type and current status
The current status comes from the CDC table `silver_work_order_current`. Hours created → completed are only measured for COMPLETED orders.

In [0]:
%sql
CREATE OR REPLACE VIEW capstone_5iot.gold.vw_maintenance
COMMENT 'Dashboard: work orders by status and type - parts cost, time to complete, linked failures'
AS
WITH orders AS (
  SELECT w.work_order_id, w.maintenance_type,
         coalesce(c.status, w.work_order_status) AS current_status,
         w.parts_cost, w.completed_ts_clean,
         (unix_timestamp(w.completed_ts_clean) - unix_timestamp(w.created_ts)) / 3600 AS hours_to_complete,
         (unix_timestamp(w.planned_start_ts)   - unix_timestamp(w.created_ts)) / 3600 AS hours_to_planned_start
  FROM capstone_5iot.silver.silver_work_orders w
  LEFT JOIN capstone_5iot.silver.silver_work_order_current c ON c.work_order_id = w.work_order_id
),
linked AS (
  SELECT work_order_id, count(*) AS linked_failures, sum(downtime_minutes) AS linked_downtime
  FROM capstone_5iot.gold.fact_maintenance_episodes
  GROUP BY work_order_id
)
SELECT o.maintenance_type,
       o.current_status                          AS work_order_status,
       count(*)                                  AS work_orders,
       round(sum(o.parts_cost), 2)               AS total_parts_cost,
       round(avg(o.parts_cost), 2)               AS avg_parts_cost,
       count(o.completed_ts_clean)               AS completed_orders,
       round(avg(o.hours_to_complete), 1)        AS avg_hours_created_to_completed,
       round(avg(o.hours_to_planned_start), 1)   AS avg_hours_created_to_planned_start,
       coalesce(sum(l.linked_failures), 0)       AS linked_failures,
       coalesce(sum(l.linked_downtime), 0)       AS linked_downtime_minutes
FROM orders o
LEFT JOIN linked l ON l.work_order_id = o.work_order_id
GROUP BY o.maintenance_type, o.current_status;

### 2i. `vw_data_quality` – data quality and pipeline health in one list
Long format: `metric_group`, `layer`, `metric_name`, `metric_value`, `detail`. It covers rows per layer, rejects per DQ rule, telemetry gaps, load-audit reconciliation and the latest Day 4 test pass rate.

In [0]:
%sql
CREATE OR REPLACE VIEW capstone_5iot.gold.vw_data_quality
COMMENT 'Dashboard: rows per layer, rejects by rule, telemetry gaps, load audit, latest test pass rate'
AS
-- rows per layer (Bronze counts come from the load audit, so analysts never need Bronze access)
SELECT 'rows_per_layer' AS metric_group, 'bronze' AS layer, bronze_table AS metric_name,
       CAST(bronze_rows AS DOUBLE) AS metric_value, concat('from silver_load_audit at ', CAST(audit_ts AS STRING)) AS detail
FROM capstone_5iot.silver.silver_load_audit
UNION ALL
SELECT 'rows_per_layer', 'silver', silver_table, CAST(silver_rows AS DOUBLE),
       concat('duplicates removed ', duplicates_removed, ', rejected ', rejected, ', reconciled ', reconciled)
FROM capstone_5iot.silver.silver_load_audit
UNION ALL
SELECT 'rows_per_layer', 'silver', 'silver_rejected_records', CAST(count(*) AS DOUBLE), 'quarantine table'
FROM capstone_5iot.silver.silver_rejected_records
UNION ALL
SELECT 'rows_per_layer', 'gold', 'fact_sensor_activity', CAST(count(*) AS DOUBLE), 'one row per valid reading'
FROM capstone_5iot.gold.fact_sensor_activity
UNION ALL
SELECT 'rows_per_layer', 'gold', 'fact_maintenance_episodes', CAST(count(*) AS DOUBLE), 'one row per failure event'
FROM capstone_5iot.gold.fact_maintenance_episodes
UNION ALL
SELECT 'rows_per_layer', 'gold', 'fact_status_transitions', CAST(count(*) AS DOUBLE), 'one row per status change'
FROM capstone_5iot.gold.fact_status_transitions
UNION ALL
SELECT 'rows_per_layer', 'gold', 'mart_sensor_daily', CAST(count(*) AS DOUBLE), 'one row per machine per day'
FROM capstone_5iot.gold.mart_sensor_daily
UNION ALL
-- rejected records per DQ rule
SELECT 'rejected_by_rule', 'silver', r.rule_id, CAST(count(*) AS DOUBLE),
       concat(r.source_table, ' - ', c.rule, ' (', c.severity, ', ', c.action, ')')
FROM capstone_5iot.silver.silver_rejected_records r
JOIN capstone_5iot.silver.dq_rule_catalogue c ON r.rule_id = c.rule_id
GROUP BY r.rule_id, r.source_table, c.rule, c.severity, c.action
UNION ALL
-- telemetry gaps (no reading for more than 24 h)
SELECT 'telemetry_gaps', 'silver', 'gaps_over_24h', CAST(count(*) AS DOUBLE), 'no reading from a machine for more than 24 h'
FROM capstone_5iot.silver.silver_telemetry_gaps
UNION ALL
SELECT 'telemetry_gaps', 'silver', 'machines_with_gaps', CAST(count(DISTINCT machine_id) AS DOUBLE), 'machines with at least one gap'
FROM capstone_5iot.silver.silver_telemetry_gaps
UNION ALL
SELECT 'telemetry_gaps', 'silver', 'max_gap_hours', max(gap_hours), 'longest gap'
FROM capstone_5iot.silver.silver_telemetry_gaps
UNION ALL
-- load audit and latest Day 4 test run
SELECT 'load_audit', 'silver', 'tables_not_reconciled', CAST(count_if(NOT reconciled) AS DOUBLE), 'bronze = silver + duplicates + rejected'
FROM capstone_5iot.silver.silver_load_audit
UNION ALL
SELECT 'test_results', 'gold', 'latest_pass_rate', round(count_if(status = 'PASS') / count(*), 4),
       concat(count_if(status = 'PASS'), ' of ', count(*), ' checks passed, run ', CAST(max(run_ts) AS STRING))
FROM capstone_5iot.gold.test_results
WHERE run_ts = (SELECT max(run_ts) FROM capstone_5iot.gold.test_results);

## Step 3 – Check the views
Every view must return rows, and the old `dashboard` schema must be gone.

In [0]:
VIEWS = ["vw_kpi_headline", "vw_daily_trend", "vw_machine_reliability", "vw_plant_summary", "vw_failure_pareto",
         "vw_sensor_drift", "vw_failure_correlation", "vw_maintenance", "vw_data_quality"]

rows = []
for v in VIEWS:
    n = spark.table(f"capstone_5iot.gold.{v}").count()
    rows.append((v, n, "PASS" if n > 0 else "FAIL"))
display(spark.createDataFrame(rows, "view string, rows long, status string"))

old_schema = spark.sql("SHOW SCHEMAS IN capstone_5iot LIKE 'dashboard'").count()
print("dashboard schema deleted:", "PASS" if old_schema == 0 else "FAIL")

In [0]:
display(spark.table("capstone_5iot.gold.vw_kpi_headline"))
display(spark.table("capstone_5iot.gold.vw_failure_pareto").orderBy("pareto_rank").limit(10))
display(spark.table("capstone_5iot.gold.vw_failure_correlation").orderBy("reading_group"))

## Step 4 – Build the AI/BI dashboard (in the UI)
Go to **Dashboards → Create dashboard** → name it **Equipment Reliability – Capstone 05**.
On the **Data** tab, add one dataset per view: `SELECT * FROM capstone_5iot.gold.<view_name>`.
The risk view `vw_machine_failure_risk` is created by notebook `7_4_ml_mlflow`. Add it after that notebook has run.

**Global filters** (add on every page, link to the datasets that have the field)

| Filter | Field | Datasets |
|---|---|---|
| Date range | `activity_date` | vw_daily_trend, vw_sensor_drift |
| Plant / region | `plant_name`, `region` | vw_machine_reliability, vw_plant_summary, vw_machine_failure_risk |
| Machine type | `machine_type` | vw_machine_reliability, vw_machine_failure_risk |
| Criticality | `criticality` | vw_machine_reliability, vw_machine_failure_risk |

The headline, Pareto, correlation and maintenance views are whole-period totals, so the filters don't change them.

**Page 1 – Executive overview**

| Widget | Type | Dataset | Fields |
|---|---|---|---|
| Overall availability | Counter (%) | vw_kpi_headline | overall_availability |
| Abnormal reading rate | Counter (%) | vw_kpi_headline | abnormal_reading_rate |
| Total downtime (min) | Counter | vw_kpi_headline | total_downtime_minutes |
| Failures | Counter | vw_kpi_headline | failure_count |
| Machines monitored | Counter | vw_kpi_headline | machines_monitored |
| MTBF (h) / MTTR (min) | 2 counters | vw_kpi_headline | mtbf_hours, mttr_downtime_minutes_per_failure |
| Fleet availability trend | Line | vw_daily_trend | x activity_date, y fleet_availability |
| Daily failures and downtime | Bar | vw_daily_trend | x activity_date, y failures, downtime_minutes |
| Abnormal rate trend | Line | vw_daily_trend | x activity_date, y abnormal_reading_rate |

**Page 2 – Machine and plant reliability**

| Widget | Type | Dataset | Fields |
|---|---|---|---|
| Top 10 machines by downtime | Bar (horizontal) | vw_machine_reliability | y machine_id, x downtime_minutes, filter downtime_rank ≤ 10, colour criticality |
| Downtime heatmap | Heatmap | vw_machine_reliability | x machine_type, y criticality, colour SUM(downtime_minutes) |
| Downtime by region | Bar | vw_plant_summary | x region, y SUM(downtime_minutes) |
| Plant table | Table | vw_plant_summary | plant_name, region, machines, failures, downtime_minutes, availability |
| Machine table | Table | vw_machine_reliability | machine_id, plant_name, machine_type, criticality, failures, downtime_minutes, availability, downtime_rank |

**Page 3 – Failure analysis**

| Widget | Type | Dataset | Fields |
|---|---|---|---|
| Downtime Pareto | Bar + line | vw_failure_pareto | x failure_code (sort pareto_rank), bar downtime_minutes, line cumulative_downtime_pct |
| Vital few codes | Counter | vw_failure_pareto | COUNT where pareto_class = 'Vital few (80% of downtime)' |
| Downtime by category | Bar | vw_failure_pareto | x failure_category, y SUM(downtime_minutes) |
| Downtime by root cause | Bar | vw_failure_pareto | x root_cause_group, y SUM(downtime_minutes) |
| Abnormal rate before failures | Bar | vw_failure_correlation | x reading_group, y abnormal_reading_rate |

**Page 4 – Sensor health and drift**

| Widget | Type | Dataset | Fields |
|---|---|---|---|
| Vibration vs 7-day baseline | Line (2 series) | vw_sensor_drift | x activity_date, y avg_vibration_mm_s and rolling_7d_avg_vibration_mm_s, machine_id filter |
| Temperature vs 7-day baseline | Line | vw_sensor_drift | avg_temperature_c, rolling_7d_avg_temperature_c |
| Drift events | Table | vw_sensor_drift | machine_id, activity_date, averages and flags, filter any_drift_flag = true |

**Page 5 – Maintenance**

| Widget | Type | Dataset | Fields |
|---|---|---|---|
| Work orders by status | Stacked bar | vw_maintenance | x work_order_status, y work_orders, colour maintenance_type |
| Total parts cost | Counter | vw_maintenance | SUM(total_parts_cost) |
| Time to complete | Table | vw_maintenance | maintenance_type, work_order_status, completed_orders, avg_hours_created_to_completed |

**Page 6 – Data quality and pipeline health**

| Widget | Type | Dataset | Fields |
|---|---|---|---|
| Rejected records | Counter | vw_data_quality | SUM(metric_value) where metric_group = 'rejected_by_rule' |
| Latest test pass rate | Counter (%) | vw_data_quality | metric_value where metric_name = 'latest_pass_rate' |
| Rejects by rule | Bar | vw_data_quality | x metric_name, y metric_value, filter metric_group = 'rejected_by_rule' |
| Rows per layer | Table | vw_data_quality | layer, metric_name, metric_value where metric_group = 'rows_per_layer' |

**Page 7 – Failure risk (ML pilot)** – after running `7_4_ml_mlflow`

| Widget | Type | Dataset | Fields |
|---|---|---|---|
| Machines by risk band | Bar | vw_machine_failure_risk | x risk_band, y COUNT |
| Risk table | Table | vw_machine_failure_risk | machine_id, plant_name, machine_type, criticality, risk_score, risk_band (sort risk_score desc) |
| Caveat | Text | – | "Pilot model trained on 1 labelled day of synthetic data – use as a relative ranking only" |

**Refresh:** the views query Gold live. Add a refresh schedule to the dashboard (after the 06:00 job), or add a dashboard task at the end of the Workflows job.